# Lesson 13 - Object-Oriented Programming

**Goal:** Use classes, objects, encapsulation, properties, inheritance, overriding, polymorphism, abstraction, and composition with clear responsibilities.

**Level:** intermediate. Complete the preceding chapters first. This notebook defines its own
inputs and runs independently. Run cells top to bottom with **Shift+Enter**. Predict results,
inspect the output, and change one input at a time. No external services or third-party packages
are required beyond your Jupyter setup.

Examples include lightweight executable checks. Practice cells intentionally contain no solutions;
expected results help you verify your own work. Temporary files and databases are isolated and cleaned up.

Where salary examples appear, use the course's simplified exercise rules: DA 48% of basic,
HRA 16% capped at 16,000 monthly, PF 12% of basic plus DA, and annual tax 20% only on annual
gross above 12,00,000, divided by 12 for monthly tax. PF is a separate deduction.
These are learning assumptions, not actual tax or payroll rules.

## 1. Define an instance

A class defines a type. __init__ initializes an instance; self refers to that instance. Instance attributes belong to one employee. Validate state when constructing the object.

In [1]:
class Employee:
    def __init__(self, employee_id, name, basic_pay):
        if basic_pay < 0:
            raise ValueError("Basic pay cannot be negative")
        self.employee_id = employee_id
        self.name = name
        self.basic_pay = basic_pay

    def monthly_da(self):
        return self.basic_pay * 0.48

asha = Employee(101, "Asha", 52_500)
arun = Employee(102, "Arun", 75_000)
print(asha.name, asha.monthly_da())
print(arun.name, arun.monthly_da())
assert asha.monthly_da() == 25_200

Asha 25200.0
Arun 36000.0


## 2. Instance state versus shared state

A mutable class attribute is shared across instances. Put per-employee notes in __init__. A class constant can hold a shared immutable default. Plain attributes can still be reassigned; validation at construction alone is not permanent protection.

In [2]:
class ReviewRecord:
    DEFAULT_STATUS = "pending"

    def __init__(self, employee_id):
        self.employee_id = employee_id
        self.status = self.DEFAULT_STATUS
        self.notes = []

first = ReviewRecord(101)
second = ReviewRecord(102)
first.notes.append("Ready")
print(first.notes, second.notes)
assert second.notes == []

['Ready'] []


## 3. Composition

Composition means one object uses another object. Keep employee identity separate from a salary policy that can be replaced. Inheritance is useful for genuine substitutable types, but it is not required for sharing a formula.

In [3]:
class SalaryPolicy:
    def __init__(self, da_rate=0.48):
        self.da_rate = da_rate

    def calculate_da(self, basic):
        return basic * self.da_rate

class Payslip:
    def __init__(self, employee, policy):
        self.employee = employee
        self.policy = policy

    def da(self):
        return self.policy.calculate_da(self.employee.basic_pay)

slip = Payslip(asha, SalaryPolicy())
print(slip.da())
assert slip.da() == 25_200

25200.0


## 4. A useful representation

__repr__ helps inspection and debugging. Return text rather than printing. Keep calculations callable without relying on display. Do not include sensitive real records in diagnostic output.

In [4]:
class Department:
    def __init__(self, code, name):
        self.code = code
        self.name = name

    def __repr__(self):
        return f"Department(code={self.code!r}, name={self.name!r})"

department = Department("FIN", "Finance")
print(department)
assert repr(department) == "Department(code='FIN', name='Finance')"

Department(code='FIN', name='Finance')


## 5. Encapsulation and validated properties

Encapsulation groups data and behavior and gives callers a clear interface. In Python,
    `_basic_pay` means an internal implementation detail by convention; it is not access control.
    Double-underscore names trigger name mangling, which helps avoid accidental subclass name
    collisions rather than providing security.

    A property looks like an attribute to callers but can validate assignments. Route construction
    through the same setter so initial and later values obey the same rule. The read-only DA
    property is derived from current basic pay, avoiding an outdated stored value.

In [5]:
import math

class PayrollEmployee:
    def __init__(self, employee_id, name, basic_pay):
        if type(employee_id) is not int or employee_id <= 0:
            raise ValueError("ID must be a positive integer")
        if not isinstance(name, str) or not name.strip():
            raise ValueError("Name must be nonblank text")
        self.employee_id = employee_id
        self.name = name.strip()
        self.basic_pay = basic_pay

    @property
    def basic_pay(self):
        return self._basic_pay

    @basic_pay.setter
    def basic_pay(self, value):
        if isinstance(value, bool) or not isinstance(value, (int, float)):
            raise TypeError("Basic pay must be an int or float")
        if not math.isfinite(value) or value < 0:
            raise ValueError("Basic pay must be finite and nonnegative")
        self._basic_pay = value

    @property
    def monthly_da(self):
        return self.basic_pay * 0.48

employee = PayrollEmployee(101, " Asha ", 52_500)
employee.basic_pay = 60_000
print(employee.name, employee.basic_pay, employee.monthly_da)
try:
    employee.basic_pay = -1
except ValueError as error:
    print(error)
assert employee.basic_pay == 60_000
assert employee.monthly_da == 28_800

Asha 60000 28800.0
Basic pay must be finite and nonnegative


## 6. Inheritance, overriding, and super()

Use inheritance for an **is-a** relationship where the subclass can satisfy the parent
    interface. A subclass inherits behavior and can override a method. `super()` delegates to
    the next implementation in the method resolution order; in this single-inheritance example,
    that is the parent. The bonus here is a separate exercise benefit, not an extra income head
    silently added to the earlier salary worksheet.

In [6]:
class BonusEmployee(PayrollEmployee):
    BONUS_RATE = 0.05

    def __init__(self, employee_id, name, basic_pay, service_months):
        super().__init__(employee_id, name, basic_pay)
        if type(service_months) is not int or service_months < 0:
            raise ValueError("Service months must be a nonnegative integer")
        self.service_months = service_months

    def bonus(self):
        if self.service_months >= 12:
            return self.basic_pay * self.BONUS_RATE
        return 0.0

class SeniorBonusEmployee(BonusEmployee):
    def bonus(self):
        return super().bonus() * 2

regular = BonusEmployee(102, "Arun", 60_000, 12)
senior = SeniorBonusEmployee(103, "Neha", 60_000, 12)
print(regular.bonus(), senior.bonus())
print(isinstance(senior, PayrollEmployee))
assert regular.bonus() == 3_000
assert senior.bonus() == 6_000

3000.0 6000.0
True


## 7. Polymorphism and duck typing

Polymorphism lets the same operation work with different implementations. Python often
    uses duck typing: an object can participate if it supports the required behavior, without
    inheriting from a particular class. Keep the contract consistent: these policies accept
    monthly basic pay and return a monthly benefit in the same units.

In [7]:
class PercentageBenefit:
    def amount(self, basic_pay):
        return basic_pay * 0.05

class FixedBenefit:
    def amount(self, basic_pay):
        return 1_000.0

def benefit_total(policies, basic_pay):
    return sum(policy.amount(basic_pay) for policy in policies)

benefits = [PercentageBenefit(), FixedBenefit()]
for policy in benefits:
    print(type(policy).__name__, policy.amount(60_000))
assert benefit_total(benefits, 60_000) == 4_000

PercentageBenefit 3000.0
FixedBenefit 1000.0


## 8. Abstraction with an abstract base class

An abstract base class documents a required interface and prevents construction when
    abstract methods are unimplemented. It does not automatically validate the meaning of
    return values. Use it when an explicit shared contract helps; duck typing is also valid.
    Prefer composition when behavior needs to be swapped independently of employee identity.

In [8]:
from abc import ABC, abstractmethod

class BenefitPolicy(ABC):
    @abstractmethod
    def amount(self, basic_pay):
        """Return a nonnegative monthly benefit."""
        raise NotImplementedError

class CappedBenefit(BenefitPolicy):
    def amount(self, basic_pay):
        return min(basic_pay * 0.05, 2_000.0)

try:
    BenefitPolicy()
except TypeError:
    print("Implement the abstract method before constructing a policy.")

policy = CappedBenefit()
assert policy.amount(60_000) == 2_000
assert benefit_total([policy, FixedBenefit()], 60_000) == 3_000

Implement the abstract method before constructing a policy.


## 9. Instance, class, and static methods

An instance method receives `self` and uses one object's state. A class method receives
    `cls`, which makes it useful for alternate constructors that honor subclasses. A static
    method has no automatic self or cls; it is a helper closely related to the class. A module
    function is often equally suitable for such a helper. Class attributes such as BONUS_RATE
    are shared defaults; put mutable per-instance state in __init__.

In [9]:
class EmployeeRecord(PayrollEmployee):
    @classmethod
    def from_record(cls, record):
        return cls(record["id"], record["name"], record["basic"])

    @staticmethod
    def is_valid_code(code):
        return (isinstance(code, str) and len(code) == 8
                and code.startswith("EMP-")
                and code[4:].isascii() and code[4:].isdigit())

record_employee = EmployeeRecord.from_record({"id": 104, "name": "Rina", "basic": 75_000})
print(record_employee.name)
assert isinstance(record_employee, EmployeeRecord)
assert EmployeeRecord.is_valid_code("EMP-1042")
assert not EmployeeRecord.is_valid_code("EMP-12")

Rina


## 10. Special methods and readable objects

`__repr__` supports debugging; `__str__` supports readable display. Return strings from
    these methods. You can implement equality, but define what equality means first. Mutable
    employee objects below retain identity equality; two matching-looking objects are not
    automatically the same object. Immutable value records and dataclass equality appear in
    Lesson 16. Keep inheritance shallow and avoid adding special methods without a use case.

In [10]:
class DisplayEmployee(PayrollEmployee):
    def __repr__(self):
        return f"DisplayEmployee(employee_id={self.employee_id!r}, name={self.name!r}, basic_pay={self.basic_pay!r})"

    def __str__(self):
        return f"{self.employee_id}: {self.name}"

display_employee = DisplayEmployee(105, "Dev", 50_000)
print(str(display_employee))
print(repr(display_employee))
assert str(display_employee) == "105: Dev"

105: Dev
DisplayEmployee(employee_id=105, name='Dev', basic_pay=50000)


## Practice lab

Define all inputs in each answer cell, use `practice_` names where helpful, and explain important
decisions in comments. Try a task before looking at its expected results. Add at least one normal,
one boundary, and one invalid-input check where the task has an input contract.
Keep file exercises inside TemporaryDirectory until you deliberately choose a permanent output path.

### Exercise 1. Validated employee

Extend Employee to reject a blank name and test negative basic pay.

**Check:** Blank and negative cases raise ValueError; zero basic remains valid.

In [11]:
# Write your solution and checks here.

### Exercise 2. Independent notes

Create two review objects and append a note to only one.

**Check:** The other object still has an empty notes list.

In [12]:
# Write your solution and checks here.

### Exercise 3. Salary policy

Add capped HRA and PF methods to SalaryPolicy.

**Check:** Basic 120,000: HRA 16,000; PF on basic plus DA 21,312.

In [13]:
# Write your solution and checks here.

### Exercise 4. Compose a payslip

Return gross, deductions, and net from Payslip using an injected policy.

**Check:** 75,000 basic and 1,500 electricity gives net 106,280.

In [14]:
# Write your solution and checks here.

### Exercise 5. Decide whether a class helps

Compare a one-off pure DA function with a class holding an employee and configurable policy.

**Check:** Explain what state or responsibility justifies the class.

In [15]:
# Write your solution and checks here.

### Exercise 6. Validated property

Test construction and reassignment with -1, NaN, infinity, True, numeric text, zero, and a positive number. Confirm failed assignments preserve the old value.

**Check:** Zero and positive numbers are accepted. Text and booleans raise TypeError; negative and nonfinite numbers raise ValueError.

In [16]:
# Write your solution and boundary checks here.

### Exercise 7. Subclass and override

Add a subclass that awards a 7% benefit only after 24 months. Reuse parent initialization with super(). Keep benefits separate from salary tax.

**Check:** Basic 60,000: 23 months gives zero; 24 months gives 4,200.

In [17]:
# Write your solution and boundary checks here.

### Exercise 8. Interchangeable policies

Write a function accepting any object with amount(basic). Test FixedBenefit, PercentageBenefit, and CappedBenefit through the same function.

**Check:** At basic 60,000 the three amounts are 1,000; 3,000; 2,000.

In [18]:
# Write your solution and boundary checks here.

### Exercise 9. Abstract contract

Create an incomplete BenefitPolicy subclass and a complete one. Catch the construction error for the incomplete class.

**Check:** Only the complete implementation can be instantiated.

In [19]:
# Write your solution and boundary checks here.

### Exercise 10. Alternative constructor

Use from_record on an EmployeeRecord subclass and confirm the returned type honors cls.

**Check:** The result is an instance of your subclass, not a hard-coded parent type.

In [20]:
# Write your solution and boundary checks here.

### Exercise 11. Composition versus inheritance

Use an employee and a replaceable policy inside a payslip object. Explain why the payslip is not a subclass of the employee.

**Check:** Changing the policy changes the benefit while employee identity stays the same.

In [21]:
# Write your solution and boundary checks here.

## Chapter project

Build two employees and two policy objects, generate payslips without printing inside calculations, and render the returned values separately. Keep per-instance lists independent.

Extend the project with validated salary properties and two interchangeable policies. Demonstrate one justified subclass and an alternate constructor. Test rejected updates and verify that original salary formulas remain unchanged.

In [22]:
# Build your chapter project here. Separate inputs, logic, and display.

## Reflection

1. Which input assumptions does your solution enforce, and which does it document?
2. What happens for empty input, a boundary value, or a malformed record?
3. Which parts can be reused without the notebook?
4. What would make the code clearer to another learner?

Restart the kernel and run all cells after completing your work to catch hidden dependencies.

**Next:** `14_iterators_and_generators.ipynb` - Iterators and Generators.